# 00 · Prepare THINGS-EEG2 and the CLIP targets

The video transformer (`01`) needs two things:

1. **THINGS-EEG2** (Gifford et al., 2022, *NeuroImage*; OSF project `3jk45`): 10 participants saw 16,540 training images 4 times and 200 test images 80 times, 100 ms per image. We use the official **preprocessed release** (OSF component `anp5v`): 17 occipital/parietal channels (the same as ours, in the same order), 100 Hz, baseline-corrected and whitened. It is public and older than the hackathon.
2. **One target space for both datasets**: CLIP ViT-B/32 image embeddings (`openai/clip-vit-base-patch32`) of the THINGS images and of our video frames. The EEG model learns EEG → CLIP on THINGS and keeps the same output space on our data, so even the output layer transfers.

**Downloads** (~11 GB): 10 × `sub-XX.zip` (17-channel EEG) and the image set (OSF `y63gw`). Every file is checked against the SHA-256 that the OSF API reports before it is used. From the challenge data only the **videos** are read (no EEG, no labels). Rerunning skips everything that already exists.

**Output** (GPU server):
```
~/eeg-data/things-eeg2/compact/sub-XX_train.npy    float32 (16540 images, 4 repetitions, 17 channels, 80 samples = 0-0.8 s)
                       compact/sub-XX_test.npy     float32 (200, 80, 17, 80)
                       compact/info.json           channel names, times, sampling rate
                       clip_vit_b32/things_train.npy, things_test.npy   CLIP of the images (16540, 512), (200, 512)
~/eeg-data/eeg-video-embeddings/clip_vit_b32/embeddings.npz   video ID -> (32 frames, 512), + metadata.json
```

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path
import hashlib, json, os, threading, time, urllib.request, zipfile

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')   # Hugging Face's widget bars freeze at 0 % here

import av
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from PIL import Image
from tqdm import tqdm                # text bars: widget bars (tqdm.auto) freeze at 0 % through the cluster's kernel gateway
from transformers import CLIPImageProcessor, CLIPModel

transformers.utils.logging.disable_progress_bar()

## 1. Configuration

In [ ]:
DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', str(Path.home() / 'eeg-data' / 'eeg-mind-reading-challenge'))).expanduser()
if (DATA_ROOT / 'eeg-mind-reading-challenge').is_dir():
    DATA_ROOT /= 'eeg-mind-reading-challenge'
THINGS_ROOT = Path(os.environ.get('THINGS_EEG2_ROOT', str(Path.home() / 'eeg-data' / 'things-eeg2'))).expanduser()
VIDEO_CLIP_DIR = Path(os.environ.get('EEG_EMBEDDING_ROOT', str(Path.home() / 'eeg-data' / 'eeg-video-embeddings'))
                      ).expanduser() / 'clip_vit_b32'          # team folder for video embeddings, one subfolder per model
THINGS_CLIP_DIR = THINGS_ROOT / 'clip_vit_b32'

OSF_EEG_NODE = 'anp5v'               # THINGS-EEG2 "Preprocessed EEG data"
OSF_IMAGE_NODE = 'y63gw'             # THINGS-EEG2 "Image set"
THINGS_SUBJECTS = list(range(1, 11))
EEG_FILES = [f'sub-{s:02d}.zip' for s in THINGS_SUBJECTS]        # the 17-channel release
IMAGE_FILES = ['training_images.zip', 'test_images.zip', 'image_metadata.npy']
N_TRAIN_IMAGES, N_TRAIN_REPS = 16540, 4                          # dataset facts (OSF wiki)
N_TEST_IMAGES, N_TEST_REPS = 200, 80
N_CHANNELS = 17
EPOCH_START_S = 0.0                  # keep 0-0.8 s: our recordings start at video onset (no pre-stimulus interval)

CLIP_CHECKPOINT = 'openai/clip-vit-base-patch32'
CLIP_PREP = dict(resize=224, crop=224, interp='bicubic',           # preprocessor_config.json of the checkpoint,
                 mean=(0.48145466, 0.4578275, 0.40821073),         # checked against CLIPImageProcessor in section 4
                 std=(0.26862954, 0.26130258, 0.27577711))
N_VIDEO_FRAMES = 32                  # 3 s / 32 = one frame per 94 ms, about one visual response latency;
                                     # the frame grid of the team's layer-wise features (pretrained_transformers/01)
CLIP_S = 3.0                         # dataset fact: the EEG covers 3 s from video onset
N_VIDEOS = 1102                      # dataset fact: 1,000 training + 102 test videos
DOWNLOAD_THREADS = 4                 # technical: parallel downloads from OSF
CONVERT_THREADS = 5                  # technical: participants converted in parallel (~2 GB RAM each)
LOAD_THREADS = 32                    # technical: CPU threads decoding JPEGs / videos ahead of the GPUs
IMAGES_PER_BATCH = 1024              # technical: images per step, split evenly over the GPUs
VIDEOS_PER_BATCH = 32                # technical: videos (x 32 frames) per step, split evenly over the GPUs
N_GPUS = torch.cuda.device_count()
assert N_GPUS > 0, 'run this notebook on the GPU kernel'
DEVICES = [f'cuda:{g}' for g in range(N_GPUS)]

for d in [THINGS_ROOT / 'downloads', THINGS_ROOT / 'compact', THINGS_CLIP_DIR, VIDEO_CLIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('THINGS-EEG2 ->', THINGS_ROOT, '\nvideo embeddings ->', VIDEO_CLIP_DIR)

## 2. Download and verify

The OSF API lists each file with its download link and SHA-256. A file is downloaded to `*.part`, verified, and renamed only if the checksum matches, so a broken download is never used. One text bar counts the bytes of all parallel downloads.

In [ ]:
def osf_files(node):
    """name -> dict(url, sha256, size) for all files of an OSF component (follows the API pagination)."""
    url, files = f'https://api.osf.io/v2/nodes/{node}/files/osfstorage/', {}
    while url:
        with urllib.request.urlopen(url) as r:
            page = json.load(r)
        for item in page['data']:
            att = item['attributes']
            if att['kind'] == 'file':
                files[att['name']] = dict(url=item['links']['download'], sha256=att['extra']['hashes']['sha256'],
                                          size=att['size'])
        url = page['links'].get('next')
    return files


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    return h.hexdigest()


def download(name, info, bar, lock):
    target = THINGS_ROOT / 'downloads' / name
    if target.is_file():
        with lock:
            bar.update(info['size'])
        return f'{name}: present'
    partial, seen = target.with_name(name + '.part'), [0]

    def hook(blocks, block_size, _):
        now = min(blocks * block_size, info['size'])
        with lock:
            bar.update(now - seen[0])
        seen[0] = now

    urllib.request.urlretrieve(info['url'], partial, reporthook=hook)
    assert sha256(partial) == info['sha256'], f'{name}: checksum mismatch; delete {partial} and rerun'
    partial.rename(target)
    return f'{name}: {info["size"] / 1e9:.2f} GB verified'


listing = osf_files(OSF_EEG_NODE) | osf_files(OSF_IMAGE_NODE)
files = EEG_FILES + IMAGE_FILES
assert all(n in listing for n in files), f'not on OSF: {[n for n in files if n not in listing]}'
lock = threading.Lock()
bar = tqdm(total=sum(listing[n]['size'] for n in files), desc='Download', unit='B', unit_scale=True)
with ThreadPoolExecutor(DOWNLOAD_THREADS) as pool:
    messages = list(pool.map(lambda n: download(n, listing[n], bar, lock), files))
bar.close()
print('\n'.join(messages))

## 3. Extract the EEG and find the image order

- Each `sub-XX.zip` holds `preprocessed_eeg_training.npy` and `preprocessed_eeg_test.npy`. These are **pickled Python dictionaries**, readable only with `allow_pickle=True`; that is safe here only because the files were verified against the official OSF checksums. We keep 0–0.8 s (80 samples) as float32.
- `image_metadata.npy` (also a verified pickle) lists the image files in the order of the EEG conditions: condition *i* = image *i*.

In [ ]:
def extract(zip_name, dest):
    marker = dest / f'.extracted_{zip_name}'          # one marker per archive: two archives can share a folder
    if not marker.is_file():
        with zipfile.ZipFile(THINGS_ROOT / 'downloads' / zip_name) as z:
            z.extractall(dest)
        marker.touch()
    return dest


def find(folder, name):
    hits = sorted(folder.rglob(name))
    assert len(hits) == 1, f'expected one {name} in {folder}, found {hits}'
    return hits[0]


def convert(s):
    """Unzip one participant, store 0-0.8 s as float32; returns its channel names, times and sampling rate."""
    out_tr, out_te = THINGS_ROOT / 'compact' / f'sub-{s:02d}_train.npy', THINGS_ROOT / 'compact' / f'sub-{s:02d}_test.npy'
    info_path = THINGS_ROOT / 'compact' / f'sub-{s:02d}_info.json'
    if out_tr.is_file() and out_te.is_file() and info_path.is_file():
        return json.loads(info_path.read_text())
    folder = extract(f'sub-{s:02d}.zip', THINGS_ROOT / 'preprocessed' / f'sub-{s:02d}')
    infos = []
    for split, out, shape in [('training', out_tr, (N_TRAIN_IMAGES, N_TRAIN_REPS)),
                              ('test', out_te, (N_TEST_IMAGES, N_TEST_REPS))]:
        d = np.load(find(folder, f'preprocessed_eeg_{split}.npy'), allow_pickle=True).item()
        times = np.asarray(d['times'])
        keep = times >= EPOCH_START_S - 1e-9
        infos.append(dict(ch_names=list(d['ch_names']), times_s=np.round(times[keep], 4).tolist(),
                          sfreq=float(round(1 / np.median(np.diff(times))))))
        data = np.asarray(d['preprocessed_eeg_data'])
        assert data.shape[:3] == (*shape, N_CHANNELS), f'sub-{s:02d} {split}: unexpected shape {data.shape}'
        np.save(out, data[..., keep].astype(np.float32))
    assert infos[0] == infos[1], f'sub-{s:02d}: training and test files differ in channels or times'
    info_path.write_text(json.dumps(infos[0]))
    return infos[0]


with ThreadPoolExecutor(CONVERT_THREADS) as pool:
    infos = list(tqdm(pool.map(convert, THINGS_SUBJECTS), total=len(THINGS_SUBJECTS), desc='Unzip + convert EEG',
                      unit='participant'))
assert all(i == infos[0] for i in infos), 'channels or times differ between participants'
INFO = infos[0] | dict(source='THINGS-EEG2 preprocessed release, OSF anp5v (Gifford et al., 2022)',
                       epoch_start_s=EPOCH_START_S, created=str(date.today()))
(THINGS_ROOT / 'compact' / 'info.json').write_text(json.dumps(INFO, indent=1))
print(f'{len(INFO["ch_names"])} channels: {INFO["ch_names"]}')
print(f'{len(INFO["times_s"])} samples at {INFO["sfreq"]:.0f} Hz: {INFO["times_s"][0]} ... {INFO["times_s"][-1]} s')

meta = np.load(THINGS_ROOT / 'downloads' / 'image_metadata.npy', allow_pickle=True).item()
image_dir = THINGS_ROOT / 'images'
for z in ['training_images.zip', 'test_images.zip']:
    extract(z, image_dir)
by_name = {}
for p in image_dir.rglob('*.jpg'):
    assert p.name not in by_name, f'duplicate image name {p.name}'
    by_name[p.name] = p
TRAIN_IMAGES = [by_name[f] for f in meta['train_img_files']]
TEST_IMAGES = [by_name[f] for f in meta['test_img_files']]
assert len(TRAIN_IMAGES) == N_TRAIN_IMAGES and len(TEST_IMAGES) == N_TEST_IMAGES

## 4. CLIP embeddings on all GPUs

The same model and preprocessing for the THINGS images and our video frames: resize the short side to 224 (bicubic, antialiased), centre crop 224, CLIP normalisation, as `CLIPImageProcessor` does, but on the GPU. The first cell checks this against the official processor. CPU threads decode the next batch while the GPUs embed the current one. Embeddings are stored without L2 normalisation (`01` normalises after averaging frames).

**Video frames**: the frame on screen at 0, 3/32, 2·3/32, … s, decoded with PyAV (`metadata_errors='ignore'`: some files have non-UTF-8 metadata).

In [ ]:
# One CLIP copy per GPU, loaded one after another (from_pretrained is not thread-safe)
CLIPS = {d: CLIPModel.from_pretrained(CLIP_CHECKPOINT).eval().to(d) for d in DEVICES}


def gpu_preprocess(image, cfg, device):
    """uint8 (H, W, 3) -> normalised float32 (3, crop, crop) on the GPU: the steps of the Hugging Face processor."""
    x = torch.from_numpy(image).to(device).permute(2, 0, 1)[None].float() / 255
    h, w = x.shape[-2:]
    short = cfg['resize']
    size = (short, int(short * w / h)) if h <= w else (int(short * h / w), short)   # same rounding as Hugging Face
    x = F.interpolate(x, size=size, mode=cfg['interp'], antialias=True, align_corners=False).clamp(0, 1)
    top, left = (x.shape[-2] - cfg['crop']) // 2, (x.shape[-1] - cfg['crop']) // 2
    x = x[0, :, top:top + cfg['crop'], left:left + cfg['crop']]
    mean = torch.tensor(cfg['mean'], device=device)[:, None, None]
    std = torch.tensor(cfg['std'], device=device)[:, None, None]
    return (x - mean) / std


@torch.inference_mode()
def gpu_embed(images, device):
    x = torch.stack([gpu_preprocess(im, CLIP_PREP, device) for im in images])
    return CLIPS[device].get_image_features(pixel_values=x).float().cpu().numpy()


def embed_items(items, load, per_batch, desc):
    """load(item) -> list of uint8 RGB arrays (one image, or the frames of a video); returns one (k, 512) array per
    item. CPU threads load the next batch while the GPUs embed the current one, an equal share each."""
    out = [None] * len(items)
    chunks = [list(range(i, min(i + per_batch, len(items)))) for i in range(0, len(items), per_batch)]
    bar = tqdm(total=len(items), desc=desc, unit='item')
    with ThreadPoolExecutor(LOAD_THREADS) as loaders, ThreadPoolExecutor(N_GPUS) as gpus:
        pending = [loaders.submit(load, items[i]) for i in chunks[0]]
        for k, chunk in enumerate(chunks):
            loaded = [f.result() for f in pending]
            if k + 1 < len(chunks):
                pending = [loaders.submit(load, items[i]) for i in chunks[k + 1]]
            parts = np.array_split(np.arange(len(chunk)), N_GPUS)

            def work(g):
                if len(parts[g]) == 0:
                    return
                emb = gpu_embed([im for j in parts[g] for im in loaded[j]], DEVICES[g])
                pos = 0
                for j in parts[g]:
                    out[chunk[j]] = emb[pos:pos + len(loaded[j])]
                    pos += len(loaded[j])

            list(gpus.map(work, range(N_GPUS)))
            bar.update(len(chunk))
    bar.close()
    return out


def load_image(path):
    return [np.asarray(Image.open(path).convert('RGB'))]


# Check: GPU preprocessing = official CLIPImageProcessor (small differences: PIL vs torch resampling)
check = [load_image(p)[0] for p in TEST_IMAGES[:8]]
ours = torch.stack([gpu_preprocess(im, CLIP_PREP, DEVICES[0]) for im in check]).cpu()
ref = CLIPImageProcessor.from_pretrained(CLIP_CHECKPOINT)(images=[Image.fromarray(im) for im in check],
                                                          return_tensors='pt')['pixel_values']
r = np.corrcoef(ours.numpy().ravel(), ref.numpy().ravel())[0, 1]
print(f'GPU preprocessing vs CLIPImageProcessor: correlation {r:.5f}')
assert ours.shape == ref.shape and r > 0.99

In [ ]:
t0 = time.time()
for name, paths in [('things_test', TEST_IMAGES), ('things_train', TRAIN_IMAGES)]:
    out = THINGS_CLIP_DIR / f'{name}.npy'
    if not out.is_file():
        np.save(out, np.concatenate(embed_items(paths, load_image, IMAGES_PER_BATCH, f'CLIP {name}')))
    print(name, np.load(out).shape, f'{time.time() - t0:.0f} s')


def video_frames(path, n=N_VIDEO_FRAMES):
    """The frame on screen at each of n equally spaced times over CLIP_S, decoded at CLIP's short side."""
    frames, times = [], []
    with av.open(str(path), metadata_errors='ignore') as container:
        for f in container.decode(container.streams.video[0]):
            scale = CLIP_PREP['resize'] / min(f.width, f.height)
            frames.append(f.reformat(width=round(f.width * scale), height=round(f.height * scale),
                                     format='rgb24').to_ndarray())
            times.append(float(f.pts * f.time_base))
    order = np.argsort(times)
    times = np.asarray(times)[order] - min(times)
    t = np.arange(n) * CLIP_S / n
    idx = np.clip(np.searchsorted(times, t + 1e-6, side='right') - 1, 0, len(frames) - 1)
    return [frames[order[i]] for i in idx]


video_out = VIDEO_CLIP_DIR / 'embeddings.npz'
if not video_out.is_file():
    paths = {int(p.stem): p for folder in ['stimuli_train', 'stimuli_test']
             for p in (DATA_ROOT / folder).glob('*.mp4') if p.stem.isdigit()}
    assert len(paths) == N_VIDEOS
    ids = sorted(paths)
    emb = embed_items([paths[v] for v in ids], video_frames, VIDEOS_PER_BATCH, 'CLIP video frames')
    np.savez(video_out.with_name('embeddings.tmp.npz'), **{str(v): e.astype(np.float32) for v, e in zip(ids, emb)})
    os.replace(video_out.with_name('embeddings.tmp.npz'), video_out)        # a complete file or none
with np.load(video_out) as z:
    print('our videos:', len(z.files), 'x', z[z.files[0]].shape, f'{time.time() - t0:.0f} s')

common = dict(source=f'Hugging Face {CLIP_CHECKPOINT}', revision=getattr(CLIPS[DEVICES[0]].config, '_commit_hash', None),
              output='CLIPModel.get_image_features (512-d, not L2-normalised)',
              preprocessing=CLIP_PREP | dict(where='GPU, checked against CLIPImageProcessor'),
              transformers=transformers.__version__, torch=torch.__version__, created=str(date.today()),
              generator='pretrained_transformers/abgabe/00_prepare_things_eeg2.ipynb')
(VIDEO_CLIP_DIR / 'metadata.json').write_text(json.dumps(common | dict(
    model='clip_vit_b32', n_frames=N_VIDEO_FRAMES, n_videos=N_VIDEOS, dim=512,
    frame_times_s=(np.arange(N_VIDEO_FRAMES) * CLIP_S / N_VIDEO_FRAMES).tolist(),
    frames='frame on screen at each time (start of each of 32 slots over 3 s)', decoder=f'PyAV {av.__version__}'),
    indent=1))
(THINGS_CLIP_DIR / 'metadata.json').write_text(json.dumps(common | dict(
    rows='EEG image-condition order of image_metadata.npy (things_train: 16540, things_test: 200)'), indent=1))
print('saved', VIDEO_CLIP_DIR, 'and', THINGS_CLIP_DIR)